# Fine-tuning DINOv2 on Food-101 🍽️

This notebook fine-tunes a **DINOv2-Base Vision Transformer** (`facebook/dinov2-base`) for 101-way food image classification on the [Food-101](https://huggingface.co/datasets/food101) dataset, then evaluates it and runs a few qualitative predictions.

**Environment:** Kaggle GPU notebook · 🤗 Transformers `Trainer` API.

| | |
|---|---|
| **Backbone** | DINOv2-Base (ViT) |
| **Dataset** | Food-101 — 75,750 train / 25,250 validation · 101 classes |
| **Training** | 3 epochs · lr 2e-5 · batch size 32 · FP16 mixed precision |
| **Result** | **92.35% top-1 accuracy** on a 2,000-image sample of the validation split |

> Training was run on the full dataset over multiple Kaggle sessions, resuming from a checkpoint to work around the GPU time limit.


## 1. Fine-tune on the full Food-101 dataset

Load the dataset, build the `id2label`/`label2id` maps, preprocess the images for the ViT, and launch training with the 🤗 `Trainer`.


In [16]:
# ============================================================
# 1. Install required libraries
# ============================================================
print("--- Step 1: Installing libraries ---")
!pip install -q transformers datasets accelerate bitsandbytes evaluate

# ============================================================
# 2. Imports
# ============================================================
print("\n--- Step 2: Importing libraries ---")
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification, TrainingArguments, Trainer
from transformers.data import DefaultDataCollator
import numpy as np
import evaluate

# ============================================================
# 3. Load Dataset (food-101) - FULL DATASET MODE
# ============================================================
print("\n--- Step 3: Loading full food-101 dataset (75,750 train images) ---")
dataset = load_dataset("food101")

# *** FIX: Using the FULL dataset splits for maximum accuracy ***
train_dataset = dataset['train']          # Full 75,750 images
test_dataset = dataset['validation']      # Full 25,250 images

print(f"Using {len(train_dataset)} training images and {len(test_dataset)} validation images.")

# ============================================================
# 4. Prepare Labels
# ============================================================
print("\n--- Step 4: Preparing labels ---")
labels = dataset['train'].features['label'].names
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

print(f"Number of labels: {num_labels}")

# ============================================================
# 5. Load Processor and Model
# ============================================================
print("\n--- Step 5: Loading processor and model ---")
model_checkpoint = "facebook/dinov2-base"
image_processor = AutoImageProcessor.from_pretrained(model_checkpoint)

model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label,
    ignore_mismatched_sizes=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model loaded on: {device}")

# ============================================================
# 6. Preprocessing Function (Single-Core, Stable Version)
# ============================================================
print("\n--- Step 6: Applying preprocessing (single-core, stable mode) ---")

def preprocess_fn(examples):
    inputs = image_processor(images=examples['image'], return_tensors="pt")
    inputs['labels'] = examples['label']
    return inputs

# Apply the preprocessing on a single core (stable).
train_dataset_processed = train_dataset.map(
    preprocess_fn,
    batched=True,
    remove_columns=train_dataset.column_names
)

test_dataset_processed = test_dataset.map(
    preprocess_fn,
    batched=True,
    remove_columns=test_dataset.column_names
)

# Set format for the Trainer
train_dataset_processed.set_format('torch')
test_dataset_processed.set_format('torch')

print("Preprocessing complete.")

# ============================================================
# 7. Evaluation Metric
# ============================================================
print("\n--- Step 7: Loading evaluation metric ---")
accuracy = evaluate.load("accuracy")

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return accuracy.compute(predictions=preds, references=p.label_ids)


print("\n--- Step 8: Setting training arguments ---")
training_args = TrainingArguments(
    output_dir="./dinov2_food101_model_FULL",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=500,
    eval_steps=500,
    logging_steps=100,
    learning_rate=2e-5,
    num_train_epochs=3, # Note: This will now run 3 full epochs over 75,750 images.
    weight_decay=0.01,
    fp16=True,
    push_to_hub=False,
    report_to="none",
)


# ============================================================
# 9. Trainer
# ============================================================
print("\n--- Step 9: Initializing Trainer ---")
data_collator = DefaultDataCollator()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset_processed,
    eval_dataset=test_dataset_processed,
    tokenizer=image_processor,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# ============================================================
# 10. Start Training & Final Steps
# ============================================================
print("\n--- Step 10: Starting FULL TRAINING! (This will take several hours) ---")
trainer.train()

print("\n--- Training complete! ---")

# --- FINAL EVALUATION & SAVE ---
print("Evaluating the final model...")
eval_results = trainer.evaluate()

print("\nEvaluation results:")
print(eval_results)


# Define a path for your final model
final_model_path = "./my_final_dinov2_food101_model_FULL"

print(f"Saving the final model to {final_model_path}...")
trainer.save_model(final_model_path)
image_processor.save_pretrained(final_model_path)

print("Model and processor saved!")


# Zip the model for download (as you already did)
!zip -r dinov2_food101_model_FULL.zip {final_model_path}

--- Step 1: Installing libraries ---
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 5.9 MB/s eta 0:00:0000:0100:01mm
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 31.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.2 MB/s eta 0:00:00:0

preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of Dinov2ForImageClassification were not initialized from the model checkpoint at facebook/dinov2-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Preprocessing complete.

--- Step 7: Loading evaluation metric ---



--- Step 8: Setting training arguments ---

--- Step 9: Initializing Trainer ---

--- Step 10: Starting FULL TRAINING! (This will take several hours) ---


/tmp/ipykernel_48/4076218619.py:125: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss,Validation Loss,Accuracy
500,0.920700,0.793011,0.784396
1000,0.693400,0.579619,0.840000
1500,0.658600,0.439878,0.873030
2000,0.586600,0.433154,0.878970
2500,0.317400,0.366694,0.895921
3000,0.306200,0.378070,0.889465
3500,0.293900,0.342874,0.903248
4000,0.268700,0.332052,0.904198
4500,0.229400,0.308594,0.911802
5000,0.088500,0.287116,0.919010


KeyboardInterrupt: 

## 2. Resume training from the last checkpoint

Kaggle caps GPU session length, so training is resumed from the most recent saved checkpoint to finish all 3 epochs.


In [17]:
# --- 4. The RESTART Command ---
# This tells the Trainer to load the model and optimizer state from the last full save
CHECKPOINT_PATH = "./dinov2_food101_model_FULL/checkpoint-5000"

print(f"\n--- Step 10: Restarting training from {CHECKPOINT_PATH} ---")
trainer.train(resume_from_checkpoint=CHECKPOINT_PATH) 

print("\n--- Training complete! ---")


--- Step 10: Restarting training from ./dinov2_food101_model_FULL/checkpoint-5000 ---


Step,Training Loss,Validation Loss,Accuracy
5500,0.074200,0.289738,0.920356
6000,0.065100,0.282779,0.922970
6500,0.048700,0.265351,0.928950
7000,0.045200,0.257699,0.931327



--- Training complete! ---


## 3. Save the fine-tuned model

Persist the model + image processor and zip them for download.


In [18]:
# --- FINAL SAVE AND ZIP ---

# Define a path for your final model (as defined in your script)
final_model_path = "./my_final_dinov2_food101_model_FULL"

print(f"Saving the final model to {final_model_path}...")
# This saves the model and its configuration
trainer.save_model(final_model_path)
image_processor.save_pretrained(final_model_path)

# Zip the model for easy download
!zip -r dinov2_food101_model_FULL.zip {final_model_path}

print("\nModel saved and zipped! You can now download 'dinov2_food101_model_FULL.zip' from the Output sidebar.")

Saving the final model to ./my_final_dinov2_food101_model_FULL...
  adding: my_final_dinov2_food101_model_FULL/ (stored 0%)
  adding: my_final_dinov2_food101_model_FULL/preprocessor_config.json (deflated 48%)
  adding: my_final_dinov2_food101_model_FULL/model.safetensors (deflated 7%)
  adding: my_final_dinov2_food101_model_FULL/config.json (deflated 66%)
  adding: my_final_dinov2_food101_model_FULL/training_args.bin (deflated 51%)

Model saved and zipped! You can now download 'dinov2_food101_model_FULL.zip' from the Output sidebar.


## 4. Evaluate on the held-out validation set

Reload the saved model and measure top-1 accuracy on a random sample of the Food-101 validation split (data the model never saw during training).


In [23]:
# ============================================================
# 1. SETUP AND COPY MODEL (No Copy Needed, Files are Local)
# ============================================================
print("Installing scikit-learn and datasets...")
!pip install -q scikit-learn datasets

# --- Define Paths ---
# The correct path to the model you just trained on the full dataset.
MODEL_PATH_TO_LOAD = "./my_final_dinov2_food101_model_FULL" 
print(f"Set model path to: {MODEL_PATH_TO_LOAD}")

# --- 2. Define Evaluation Functions ---
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
from tqdm import tqdm
from datasets import load_dataset 
import os
from PIL import Image

def load_model_and_processor(model_path):
    """Load the model and processor from a simple local path."""
    print(f"Loading model and processor from {model_path}...")
    
    if not os.path.exists(model_path):
        print(f"--- FATAL ERROR --- Folder not found at {model_path}")
        print("Please ensure you ran the full training script completely.")
        return None, None
        
    model = AutoModelForImageClassification.from_pretrained(model_path)
    processor = AutoImageProcessor.from_pretrained(model_path)
    model.eval()
    print("Model loaded successfully!")
    return model, processor

def evaluate_with_food101(model, processor, num_samples=None):
    """Evaluate using Food-101 validation set."""
    print("Loading Food-101 'validation' (test) split from HuggingFace...")
    dataset = load_dataset("food101", split="validation")
    
    if num_samples:
        print(f"Selecting {num_samples} random samples for evaluation...")
        dataset = dataset.shuffle(seed=42).select(range(min(num_samples, len(dataset))))
    
    print(f"Evaluating on {len(dataset)} test images...")
    id2label = model.config.id2label
    true_labels, predicted_labels = [], []
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    
    for item in tqdm(dataset, desc="Processing images"):
        true_label = item['label']
        inputs = processor(images=item['image'], return_tensors="pt").to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            predicted_class_idx = logits.argmax(-1).item()
        
        true_labels.append(true_label)
        predicted_labels.append(predicted_class_idx)
    
    return true_labels, predicted_labels, id2label

def calculate_metrics(true_labels, predicted_labels, id2label_dict):
    """Calculate and display metrics"""
    true_labels = np.array(true_labels)
    predicted_labels = np.array(predicted_labels)
    
    accuracy = accuracy_score(true_labels, predicted_labels)
    print(f"\n{'='*60}")
    print(f"OVERALL ACCURACY: {accuracy*100:.2f}%")
    print(f"{'='*60}\n")
    
    target_names = [id2label_dict[str(i)] for i in range(len(id2label_dict))]
    present_labels = sorted(list(set(true_labels) | set(predicted_labels)))
    report_target_names = [id2label_dict[str(i)] for i in present_labels]
    
    print("DETAILED CLASSIFICATION REPORT (showing top classes)")
    if len(report_target_names) > 25:
        report_target_names = report_target_names[:25]
        present_labels = present_labels[:25]
        print("(Report truncated to first 25 classes for readability)\n")
        
    print(classification_report(
        true_labels, 
        predicted_labels, 
        labels=present_labels, 
        target_names=report_target_names, 
        zero_division=0
    ))
    return {'accuracy': accuracy}


# --- 3. RUN THE EVALUATION ---
print("\n--- Starting Model Evaluation ---")

try:
    # Load from the new working path
    model, processor = load_model_and_processor(MODEL_PATH_TO_LOAD)

    if model:
        # --- CONFIGURE YOUR TEST ---
        # Testing 2000 samples for a fast, robust check.
        NUMBER_OF_SAMPLES_TO_TEST = 2000 
        
        results = evaluate_with_food101(model, processor, num_samples=NUMBER_OF_SAMPLES_TO_TEST)

        if results:
            true_labels, predicted_labels, id2label = results
            
            if len(true_labels) > 0:
                print("\n--- Final Metrics ---")
                metrics = calculate_metrics(true_labels, predicted_labels, id2label)
            else:
                print("Evaluation finished, but no valid labels were processed.")
    else:
        print("Evaluation failed because the model could not be loaded.")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

Installing scikit-learn and datasets...
Set model path to: ./my_final_dinov2_food101_model_FULL

--- Starting Model Evaluation ---
Loading model and processor from ./my_final_dinov2_food101_model_FULL...
Model loaded successfully!
Loading Food-101 'validation' (test) split from HuggingFace...
Selecting 2000 random samples for evaluation...
Evaluating on 2000 test images...


Processing images: 100%|██████████| 2000/2000 [00:48<00:00, 41.30it/s]


--- Final Metrics ---

OVERALL ACCURACY: 92.35%

An unexpected error occurred: '0'


## 5. Qualitative predictions

Sanity-check the model on real web images. The burger is an in-distribution class (`hamburger`) and is classified correctly. *Schnitzel* and *couscous* are **not** among the 101 Food-101 classes, so the model maps them to the nearest learned dish — expected behaviour for out-of-vocabulary foods.


In [26]:
import requests
from PIL import Image
from io import BytesIO
import torch
from transformers import AutoModelForImageClassification, AutoImageProcessor

# --- 1. Load your saved model and processor ---
# CORRECT PATH: Loading the model trained on the FULL dataset
model_path = "./my_final_dinov2_food101_model_FULL" 

# Load the processor and model
try:
    inference_processor = AutoImageProcessor.from_pretrained(model_path)
    inference_model = AutoModelForImageClassification.from_pretrained(model_path)
    
    # Send model to GPU (if available)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    inference_model.to(device)
    inference_model.eval()
    
    print("Model and processor loaded from disk successfully.")

except Exception as e:
    print(f"ERROR: Could not load model from {model_path}. Please ensure the directory exists.")
    print(f"Details: {e}")
    inference_model = None

if inference_model:
    # --- 2. Define the Test Images (using stable URLs) ---
    TEST_IMAGES = {
        # New stable URL for Schnitzel
        "Schnitzel (Unseen)": "https://upload.wikimedia.org/wikipedia/commons/a/ae/Wiener-Schnitzel02.jpg",
        # New stable URL for Couscous
        "Couscous (Unseen)": "https://fortheloveofcooking.net/wp-content/uploads/2012/03/DSC_5088-1.jpg"
    }
    
    # --- 3. Run Predictions ---
    print("\n--- Running Predictions on Unseen Food ---")

    for name, img_url in TEST_IMAGES.items():
        try:
            # Note: For some hosts, a User-Agent header is required to mimic a browser
            headers = {'User-Agent': 'Mozilla/5.0'}
            response = requests.get(img_url, headers=headers, timeout=10)
            response.raise_for_status() 
            
            img = Image.open(BytesIO(response.content)).convert("RGB")
            
            # Process the image
            inputs = inference_processor(images=img, return_tensors="pt").to(device)

            # Predict
            with torch.no_grad():
                logits = inference_model(**inputs).logits
            
            prediction_id = logits.argmax(-1).item()
            predicted_label = inference_model.config.id2label[prediction_id]

            print(f"\n[Test: {name}]")
            print(f"Model prediction: {predicted_label.upper()}")
            
        except Exception as e:
            print(f"\n[Test: {name}] ERROR: Failed to process image or fetch URL.")
            print(f"Details: {e}")

Model and processor loaded from disk successfully.

--- Running Predictions on Unseen Food ---

[Test: Schnitzel (Unseen)]
Model prediction: FISH_AND_CHIPS

[Test: Couscous (Unseen)]
Model prediction: FRIED_RICE


In [4]:
import requests
from PIL import Image
from io import BytesIO

# --- 1. Model should still be loaded in memory ---
# (We don't need to load it again)

# --- 2. Get a new image from a NEW host (Pexels) ---
img_url = "https://images.pexels.com/photos/1639557/pexels-photo-1639557.jpeg" # A burger

try:
    print(f"Loading image from: {img_url}")
    response = requests.get(img_url, timeout=10)
    
    # This will check for errors
    response.raise_for_status() 
    
    # Open the image
    img = Image.open(BytesIO(response.content)).convert("RGB")
    print("Successfully loaded image.")
    
    # --- 3. Process the image and make a prediction ---
    inputs = inference_processor(images=img, return_tensors="pt").to(device)

    with torch.no_grad():
        logits = inference_model(**inputs).logits

    # --- 4. Get the result ---
    prediction_id = logits.argmax(-1).item()
    predicted_label = inference_model.config.id2label[prediction_id]

    print(f"\nModel prediction: {predicted_label.upper()}")

except Exception as e:
    print(f"An error occurred: {e}")

Loading image from: https://images.pexels.com/photos/1639557/pexels-photo-1639557.jpeg
Successfully loaded image.

Model prediction: HAMBURGER
